# 05 — Topic Modeling with BERTopic

Discover latent topics in food reviews using BERTopic (sentence-transformers + UMAP + HDBSCAN).


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
from datetime import datetime

from topic_model import ReviewTopicModel
from preprocess import clean_text_series
from utils import get_logger

logger = get_logger('05_topics')
print('Setup complete ✓')

Setup complete ✓


## 1. Load & Sample Data

In [2]:
df = pd.read_parquet(os.path.join('..', 'data', 'reviews_processed.parquet'))

# Remove very short reviews and sample
df_long = df[df['Text'].str.len() > 50].copy()
SAMPLE = min(5000, len(df_long))
df_sample = df_long.sample(SAMPLE, random_state=42).reset_index(drop=True)

# Clean text
df_sample['clean_text'] = clean_text_series(df_sample['Text'],
                                             remove_stopwords=True,
                                             lemmatize=True)

# Drop empty
df_sample = df_sample[df_sample['clean_text'].str.len() > 10].reset_index(drop=True)
docs = df_sample['clean_text'].tolist()
print(f'Using {len(docs):,} documents for topic modeling')

2026-05-04 16:36:46 | INFO     | preprocess | Cleaning 5000 text samples …


Using 5,000 documents for topic modeling


## 2. Fit BERTopic

In [3]:
topic_model = ReviewTopicModel(min_topic_size=50)
topics, probs = topic_model.fit(docs)

df_sample['topic'] = topics
topic_info = topic_model.get_topic_info()
print(f'\nDiscovered {len(topic_info) - 1} topics (excl. outlier topic -1)')
topic_info.head(15)

2026-05-04 16:38:08 | INFO     | topic_model | BERTopic initialised (min_topic_size=50, embedding=all-MiniLM-L6-v2)


2026-05-04 16:38:08 | INFO     | topic_model | Fitting BERTopic on 5000 documents …


2026-05-04 16:38:08,508 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

...

2026-05-04 16:40:53,741 - BERTopic - Representation - Completed ✓


2026-05-04 16:40:53,743 - BERTopic - Topic reduction - Reduced number of topics from 4 to 4


2026-05-04 16:40:53 | INFO     | topic_model | Found 4 topics.



Discovered 3 topics (excl. outlier topic -1)


,Topic,Count,Name,Representation,Representative_Docs
0,0,3560,0_like_taste_tea_good,"[like, taste, tea, good, flavor, product, one,...",[product great tried number different green su...
1,1,716,1_dog_food_cat_treat,"[dog, food, cat, treat, love, like, one, eat, ...",[dog turn nose dry wet food premium name welln...
2,2,648,2_coffee_cup_like_flavor,"[coffee, cup, like, flavor, taste, pod, one, g...",[love coffee favorite far want try dont like s...
3,3,76,3_popcorn_popper_pop_great,"[popcorn, popper, pop, great, taste, oil, popp...",[dans perfect popcorn note thorough recipe imp...


## 3. Interactive Topic Map

In [4]:
fig = topic_model.plot_topic_map()
fig.show()

## 4. Top Words per Topic (Bar Chart)

In [5]:
fig = topic_model.plot_barchart(top_n_topics=10)
fig.show()

## 5. Topic Similarity Heatmap

In [6]:
fig = topic_model.plot_heatmap()
fig.show()

## 6. Topic Distribution Over Time

In [7]:
timestamps = pd.to_datetime(df_sample['Time'], unit='s').tolist()
fig = topic_model.plot_topics_over_time(docs, timestamps, n_bins=20)
fig.show()

0it [00:00, ?it/s]

4it [00:00, 33.84it/s]

8it [00:00, 30.77it/s]

12it [00:00, 24.90it/s]

15it [00:00, 22.04it/s]

18it [00:00, 18.26it/s]

19it [00:00, 20.37it/s]

## 7. Manual Topic Labeling

Review the top words per topic above and assign human-readable labels.

In [8]:
# Example — update after inspecting top words:
manual_labels = {
    0: 'taste quality',
    1: 'price value',
    2: 'delivery experience',
    3: 'health benefits',
    4: 'pet food',
    5: 'coffee tea',
    6: 'snack chips',
    7: 'baby food formula',
    8: 'chocolate candy',
    9: 'packaging complaints',
}

topic_model.set_topic_labels(manual_labels)
print('Labels set. Update the dictionary above after inspecting your results.')

2026-05-04 16:41:06 | INFO     | topic_model | Set 10 custom topic labels.


Labels set. Update the dictionary above after inspecting your results.


## 8. Save Model

In [9]:
topic_model.save('bertopic_model')
print('BERTopic model saved ✓')

2026-05-04 16:41:06 | INFO     | topic_model | BERTopic saved → D:\New Project\customer-review-intelligence\models\bertopic_model


BERTopic model saved ✓


## Summary

* BERTopic discovered meaningful food-review clusters automatically.
* Topics align with real product categories: coffee, pet food, baby products, snacks.
* Time-series view shows seasonal patterns and growing review volume.

**Next**: `app/main.py` (FastAPI) and `app/dashboard.py` (Streamlit).
